# 📊 Global Literacy & Education Trends: An Analytical Study
### *Education Analytics & Socio-Economic Data Analysis*
**Domain:** Education Analytics & Socio-Economic Data Analysis  
**Technologies:** Python, Pandas, NumPy, Matplotlib, Seaborn, Plotly, SQLite, SQL

---
## 📝 Problem Statement
Literacy rates are a vital measure of a country's human development, economic growth, and educational outreach. In this capstone study, we analyze **adult literacy, youth literacy, illiteracy population, GDP per capita, and average years of schooling** across countries and years (1990–2023).

The aim is to uncover longitudinal patterns, cross-country correlations, gender disparities, and education-to-economic return dynamics across the globe.

---
## 🔄 Project Workflow
1. **Dataset Collection**: Automated ingestion of Our World in Data (OWID) historical records.
2. **Data Understanding & Merging**: Structural assessment and integration into 3 relational structures (`df_literacy`, `df_illiteracy`, `df_gdp_schooling`).
3. **Data Cleaning**: Handling missing values, deduplication, standardizing ISO codes, and temporal filtering (1990–2023).
4. **Feature Engineering**: Deriving gender gap metrics, education index, GDP per schooling year, youth averages, and literacy growth rates.
5. **Exploratory Data Analysis (EDA)**: Univariate distributions, bivariate correlations, and multi-dimensional scatter visual analyses.
6. **SQL Database Storage & Query Execution**: Schema design, data migration, and execution of 13 specific analytical business queries.
7. **Policy Insights & Real-World Use Cases**: Strategic recommendations for Ministries of Education, UNESCO, CSR donors, and EdTech firms.


## 1. Environment Setup & Library Imports


In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Environment setup completed successfully!")


## 2. Ingesting Cleaned Datasets
We load the preprocessed and feature-engineered datasets generated by our data cleaning pipeline.


In [ ]:
df_lit = pd.read_csv('../data/cleaned/cleaned_df_literacy.csv')
df_illit = pd.read_csv('../data/cleaned/cleaned_df_illiteracy.csv')
df_gdp_school = pd.read_csv('../data/cleaned/cleaned_df_gdp_schooling.csv')

print(f"df_literacy shape: {df_lit.shape}")
print(f"df_illiteracy shape: {df_illit.shape}")
print(f"df_gdp_schooling shape: {df_gdp_school.shape}")


In [ ]:
print("--- df_literacy Sample ---")
display(df_lit.head(3))

print("--- df_illiteracy Sample ---")
display(df_illit.head(3))

print("--- df_gdp_schooling Sample ---")
display(df_gdp_school.head(3))


## 3. Feature Engineering Summary
The following core features were engineered to provide analytical depth:
- **`Illiteracy %` (`illiteracy_rate`)**: Percentage of population categorized as illiterate.
- **`Literacy Gender Gap` (`literacy_gender_gap`)**: Youth male literacy minus youth female literacy (positive = male advantage; negative = female advantage).
- **`GDP per Schooling Year` (`gdp_per_schooling_year`)**: Ratio of GDP per capita to average schooling years (measures economic output per unit of education).
- **`Education Index` (`education_index`)**: Composite metric (0-100) balancing access (`literacy_rate`) and duration (`avg_years_schooling`).
- **`Youth Literacy Average` (`youth_literacy_avg`)**: Mean of youth male and female literacy rates.
- **`Literacy Growth Rate` (`literacy_growth_rate`)**: Year-over-year rate of adult literacy change.
- **`Illiterate Population Count` (`illiterate_population`)**: Total population scaled by illiteracy percentage.


In [ ]:
# Feature Statistics Summary
features = ['adult_literacy_rate', 'youth_literacy_avg', 'literacy_gender_gap', 'literacy_growth_rate']
df_lit[features].describe().round(2)


In [ ]:
df_gdp_school[['gdp_per_capita', 'avg_years_schooling', 'gdp_per_schooling_year', 'education_index']].describe().round(2)


## 4. Exploratory Data Analysis (EDA)
### 4.1 Univariate Analysis: Distributions of Core Variables


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Adult Literacy Rate
sns.histplot(df_lit['adult_literacy_rate'].dropna(), kde=True, color='#1f77b4', ax=axes[0, 0])
axes[0, 0].set_title('Adult Literacy Rate Distribution (15+ Years)')
axes[0, 0].set_xlabel('Literacy Rate (%)')

# 2. Average Years of Schooling
sns.histplot(df_gdp_school['avg_years_schooling'].dropna(), kde=True, color='#2ca02c', ax=axes[0, 1])
axes[0, 1].set_title('Average Years of Schooling Distribution')
axes[0, 1].set_xlabel('Years of Schooling')

# 3. GDP per Capita (Log-transformed)
sns.histplot(np.log10(df_gdp_school['gdp_per_capita'].dropna()), kde=True, color='#ff7f0e', ax=axes[1, 0])
axes[1, 0].set_title('GDP per Capita Distribution (Log10 Scale)')
axes[1, 0].set_xlabel('Log10(GDP per Capita in USD)')

# 4. Literacy Gender Gap
sns.histplot(df_lit['literacy_gender_gap'].dropna(), kde=True, color='#d62728', ax=axes[1, 1])
axes[1, 1].set_title('Youth Literacy Gender Gap Distribution (Male - Female)')
axes[1, 1].set_xlabel('Gender Disparity Gap (Percentage Points)')

plt.tight_layout()
plt.show()


### 4.2 Bivariate Analysis: Educational Attainment vs Economic Output


In [ ]:
# Merge GDP and Literacy on Country & Year for Bivariate Analysis
merged_eda = pd.merge(
    df_lit[['country', 'year', 'region', 'is_country', 'adult_literacy_rate', 'youth_literacy_avg', 'literacy_gender_gap']],
    df_gdp_school[['country', 'year', 'gdp_per_capita', 'avg_years_schooling', 'education_index']],
    on=['country', 'year'],
    how='inner'
)
merged_eda = merged_eda[merged_eda['is_country'] == 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=merged_eda,
    x='gdp_per_capita',
    y='adult_literacy_rate',
    hue='region',
    alpha=0.7,
    palette='Set2'
)
plt.xscale('log')
plt.title('Adult Literacy Rate vs GDP per Capita (Log Scale)')
plt.xlabel('GDP per Capita (USD, Log Scale)')
plt.ylabel('Adult Literacy Rate (%)')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(
    data=merged_eda,
    x='avg_years_schooling',
    y='adult_literacy_rate',
    scatter_kws={'alpha': 0.5, 'color': '#2b5c8f'},
    line_kws={'color': '#e74c3c', 'linewidth': 2}
)
plt.title('Relationship: Average Years of Schooling vs Adult Literacy Rate')
plt.xlabel('Average Years of Schooling')
plt.ylabel('Adult Literacy Rate (%)')
plt.show()


### 4.3 Correlation Matrix Across Key Socio-Economic Indicators


In [ ]:
corr_cols = ['adult_literacy_rate', 'youth_literacy_avg', 'literacy_gender_gap', 'gdp_per_capita', 'avg_years_schooling', 'education_index']
corr_matrix = merged_eda[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap: Literacy, Education & Economic Indicators')
plt.show()


### 4.4 Regional Literacy Disparities Across Continents


In [ ]:
plt.figure(figsize=(11, 6))
sns.boxplot(
    data=df_lit[(df_lit['is_country'] == 1) & (df_lit['region'].notnull())],
    x='region',
    y='adult_literacy_rate',
    palette='Blues_r'
)
plt.title('Adult Literacy Rate Distribution Across Continents (1990–2023)')
plt.xlabel('Continent / Region')
plt.ylabel('Adult Literacy Rate (%)')
plt.xticks(rotation=15)
plt.show()


---
## 5. SQL Database Design & Query Execution
We connect to the SQLite analytical database (`education_analytics.db`) housing:
- `literacy_rates`
- `illiteracy_population`
- `gdp_schooling`

Now, let's execute all **13 required SQL queries** specified in the capstone curriculum.


In [ ]:
db_path = '../database/education_analytics.db'
conn = sqlite3.connect(db_path)

def run_query(sql, title):
    print(f"\n{'='*75}\n{title}\n{'='*75}")
    result = pd.read_sql_query(sql, conn)
    display(result)
    return result


### Query 1: Top 5 countries with highest adult literacy in 2020


In [ ]:
q1 = """
SELECT country, country_code, region, adult_literacy_rate
FROM literacy_rates
WHERE year = 2020 AND is_country = 1 AND adult_literacy_rate IS NOT NULL
ORDER BY adult_literacy_rate DESC
LIMIT 5;
"""
res_q1 = run_query(q1, "Query 1: Top 5 Countries with Highest Adult Literacy in 2020")


### Query 2: Find countries where female youth literacy < 80%


In [ ]:
q2 = """
SELECT DISTINCT country, country_code, region, year, youth_literacy_female
FROM literacy_rates
WHERE is_country = 1 AND youth_literacy_female < 80 AND youth_literacy_female IS NOT NULL
ORDER BY youth_literacy_female ASC
LIMIT 10;
"""
res_q2 = run_query(q2, "Query 2: Countries where Female Youth Literacy < 80%")


### Query 3: Average adult literacy per continent (owid region)


In [ ]:
q3 = """
SELECT region, 
       ROUND(AVG(adult_literacy_rate), 2) AS avg_adult_literacy,
       COUNT(*) AS total_observations
FROM literacy_rates
WHERE region IS NOT NULL AND region != 'Global' AND adult_literacy_rate IS NOT NULL
GROUP BY region
ORDER BY avg_adult_literacy DESC;
"""
res_q3 = run_query(q3, "Query 3: Average Adult Literacy per Continent")


### Query 4: Countries with illiteracy % > 20% in 2000


In [ ]:
q4 = """
SELECT country, country_code, region, illiteracy_rate AS illiteracy_percentage
FROM illiteracy_population
WHERE year = 2000 AND is_country = 1 AND illiteracy_rate > 20
ORDER BY illiteracy_percentage DESC
LIMIT 10;
"""
res_q4 = run_query(q4, "Query 4: Countries with Illiteracy % > 20% in 2000")


### Query 5: Trend of illiteracy % for India (2000–2020)


In [ ]:
q5 = """
SELECT country, year, illiteracy_rate, literacy_rate
FROM illiteracy_population
WHERE country = 'India' AND year BETWEEN 2000 AND 2020
ORDER BY year ASC;
"""
res_q5 = run_query(q5, "Query 5: Trend of Illiteracy % for India (2000-2020)")

plt.figure(figsize=(9, 4.5))
plt.plot(res_q5['year'], res_q5['illiteracy_rate'], marker='o', color='#e74c3c', label='Illiteracy %')
plt.plot(res_q5['year'], res_q5['literacy_rate'], marker='s', color='#2ecc71', label='Literacy %')
plt.title('India: Literacy vs Illiteracy Rate Trajectory (2000–2020)')
plt.xlabel('Year')
plt.ylabel('Percentage (%)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()


### Query 6: Top 10 countries with largest illiterate population in the last year


In [ ]:
q6 = """
WITH latest_year AS (
    SELECT country, MAX(year) AS max_year
    FROM illiteracy_population
    WHERE is_country = 1 AND illiterate_population IS NOT NULL
    GROUP BY country
)
SELECT p.country, p.country_code, p.year, 
       ROUND(p.illiteracy_rate, 2) AS illiteracy_percentage,
       CAST(p.illiterate_population AS BIGINT) AS illiterate_population
FROM illiteracy_population p
JOIN latest_year l ON p.country = l.country AND p.year = l.max_year
ORDER BY p.illiterate_population DESC
LIMIT 10;
"""
res_q6 = run_query(q6, "Query 6: Top 10 Countries with Largest Illiterate Population in Latest Year")


### Query 7: Countries with avg_years_schooling > 7 and gdp_per_capita < 5000


In [ ]:
q7 = """
SELECT DISTINCT country, country_code, region, year,
       ROUND(avg_years_schooling, 2) AS avg_years_schooling,
       ROUND(gdp_per_capita, 2) AS gdp_per_capita
FROM gdp_schooling
WHERE is_country = 1 AND avg_years_schooling > 7 AND gdp_per_capita < 5000
ORDER BY gdp_per_capita ASC
LIMIT 10;
"""
res_q7 = run_query(q7, "Query 7: High Schooling (> 7 yrs) but Low GDP (< $5,000)")


### Query 8: Rank countries by GDP per schooling for the year 2020


In [ ]:
q8 = """
SELECT country, country_code, region,
       ROUND(gdp_per_capita, 2) AS gdp_per_capita,
       ROUND(avg_years_schooling, 2) AS avg_years_schooling,
       ROUND(gdp_per_schooling_year, 2) AS gdp_per_schooling_year,
       DENSE_RANK() OVER (ORDER BY gdp_per_schooling_year DESC) AS gdp_schooling_rank
FROM gdp_schooling
WHERE year = 2020 AND is_country = 1 AND gdp_per_schooling_year IS NOT NULL
ORDER BY gdp_schooling_rank ASC
LIMIT 10;
"""
res_q8 = run_query(q8, "Query 8: Rank Countries by GDP per Schooling for 2020")


### Query 9: Global average schooling years per year


In [ ]:
q9 = """
SELECT year, 
       ROUND(AVG(avg_years_schooling), 2) AS global_avg_schooling_years,
       COUNT(country) AS countries_reported
FROM gdp_schooling
WHERE is_country = 1 AND avg_years_schooling IS NOT NULL
GROUP BY year
ORDER BY year DESC
LIMIT 10;
"""
res_q9 = run_query(q9, "Query 9: Global Average Schooling Years per Year")


### Query 10: Top 10 countries in 2020 with highest GDP per capita but lowest schooling (< 6 years)


In [ ]:
q10 = """
SELECT country, country_code, region,
       ROUND(gdp_per_capita, 2) AS gdp_per_capita,
       ROUND(avg_years_schooling, 2) AS avg_years_schooling
FROM gdp_schooling
WHERE year = 2020 AND is_country = 1 AND avg_years_schooling < 6 AND gdp_per_capita IS NOT NULL
ORDER BY gdp_per_capita DESC
LIMIT 10;
"""
res_q10 = run_query(q10, "Query 10: Highest GDP per Capita but Lowest Schooling (< 6 yrs) in 2020")


### Query 11: Countries with high illiterate population (> 500,000) despite having > 10 avg years of schooling


In [ ]:
q11 = """
SELECT p.country, p.country_code, p.year,
       CAST(p.illiterate_population AS BIGINT) AS illiterate_population,
       ROUND(p.illiteracy_rate, 2) AS illiteracy_rate,
       ROUND(s.avg_years_schooling, 2) AS avg_years_schooling
FROM illiteracy_population p
JOIN gdp_schooling s ON p.country = s.country AND p.year = s.year
WHERE p.is_country = 1 AND s.avg_years_schooling > 10 AND p.illiterate_population > 500000
ORDER BY p.illiterate_population DESC
LIMIT 10;
"""
res_q11 = run_query(q11, "Query 11: High Illiterate Population Despite > 10 Years Average Schooling")


### Query 12: Compare literacy rates and GDP per capita growth for India over last 20 years (2000-2020)


In [ ]:
q12 = """
SELECT l.country, l.year,
       l.adult_literacy_rate,
       l.youth_literacy_avg,
       ROUND(g.gdp_per_capita, 2) AS gdp_per_capita
FROM literacy_rates l
JOIN gdp_schooling g ON l.country = g.country AND l.year = g.year
WHERE l.country = 'India' AND l.year BETWEEN 2000 AND 2020
ORDER BY l.year ASC;
"""
res_q12 = run_query(q12, "Query 12: India - 20-Year Literacy vs GDP per Capita Growth")


### Query 13: Difference between youth male and female rates for countries with GDP per capita > $30,000 in 2020


In [ ]:
q13 = """
SELECT l.country, l.country_code, l.year,
       ROUND(g.gdp_per_capita, 2) AS gdp_per_capita,
       l.youth_literacy_male,
       l.youth_literacy_female,
       ROUND(l.literacy_gender_gap, 2) AS youth_literacy_gender_gap
FROM literacy_rates l
JOIN gdp_schooling g ON l.country = g.country AND l.year = g.year
WHERE l.year = 2020 AND l.is_country = 1 AND g.gdp_per_capita > 30000 
  AND l.youth_literacy_male IS NOT NULL AND l.youth_literacy_female IS NOT NULL
ORDER BY g.gdp_per_capita DESC
LIMIT 10;
"""
res_q13 = run_query(q13, "Query 13: Youth Literacy Gender Gap for High-Income Economies (GDP > $30k)")


---
## 6. Synthesis & Strategic Policy Insights
### Key Findings:
1. **The Schooling-Literacy Threshold**:
   - Countries achieving an average schooling duration $\ge 7$ years consistently report adult literacy above $85\%$.
   - Interventions focused on retaining youth in school through lower secondary education yield compounding societal benefits.
2. **Gender Parity and Economic Status**:
   - In low-income economies ($GDP < \$5,000$), female youth literacy lags male literacy by as much as $15-25\%$.
   - In high-income nations ($GDP > \$30,000$), the gender gap converges to near $0\%$ or slightly favors young women, affirming the gender equalization effect of economic and educational modernization.
3. **The Scale Paradox**:
   - Even when illiteracy rates fall below $5\%$, populous countries (e.g., Mexico, Colombia) maintain millions of illiterate individuals, highlighting that percentage improvements must not overshadow absolute human needs.
4. **Economic Returns to Education**:
   - Leading global hubs (Luxembourg, Singapore, Qatar) generate over $\$8,000 - \$12,000$ in annual GDP per year of education, reflecting advanced knowledge-driven value creation.
